In [1]:
import re
import random
import math

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
with open('ChineseName1.txt', 'r', encoding='utf-8') as f:
    words = f.read().splitlines()

# 防御性过滤：
# 即使以后把未经清洗的原始文件直接改名为 ChineseName.txt，
# 这一行也会丢弃说明行、空行和异常行。
words = [w.strip() for w in words if re.fullmatch(r'[一-鿿]{2,3}', w.strip())]

# 原始语料按拼音排序，先打乱顺序。
random.seed(40)
random.shuffle(words)

print('名字数量:', len(words))

名字数量: 3548868


In [3]:
chars = sorted(list(set(''.join(words))))

# 汉字编号从 1 开始；0 留给姓名开始/结束的边界符 '.'
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0

# 反向映射：数字 → 字符
itos = {i: s for s, i in stoi.items()}

vocab_size = len(stoi)

print('不同汉字数:', len(chars))
print('vocab_size（汉字 + 边界符）:', vocab_size)
print('前 30 个汉字:', chars[:30])
print('映射示例:', {c: stoi[c] for c in words[0]}, "'.':", stoi['.'])


不同汉字数: 6505
vocab_size（汉字 + 边界符）: 6506
前 30 个汉字: ['一', '丁', '七', '万', '丈', '三', '上', '下', '丌', '不', '与', '丐', '丑', '专', '且', '丕', '世', '丘', '丙', '业', '丛', '东', '丝', '丞', '丢', '两', '严', '个', '丫', '中']
映射示例: {'况': 405, '汉': 2727} '.': 0


In [4]:
# 把所有名字展开成 (当前字, 下一字) 对。
# 每个长度 L 的名字会贡献 L+1 个 bigram（含开头和结尾的 '.'）。
xs, ys = [], []
for w in words:
    idxs = [0] + [stoi[c] for c in w] + [0]
    xs.extend(idxs[:-1])
    ys.extend(idxs[1:])

xs = torch.tensor(xs, dtype=torch.long)
ys = torch.tensor(ys, dtype=torch.long)
num = xs.nelement()

In [5]:
g = torch.Generator().manual_seed(2147483647)  # 固定随机种子

DimEmb = 10     # 每个字的嵌入维度
LenContext = 2  # 用前 LenContext 个字预测下一个字

# 每一层的维度：第一个固定是展平后的输入维度，最后一个是输出类别数
# 层数 = len(layer_dims) - 1
# 例如 [20, 200, 2271] 是 2 层：20 → 200 → 2271
layer_dims = [DimEmb * LenContext, 200, 200, vocab_size]

assert layer_dims[0] == DimEmb * LenContext
assert layer_dims[-1] == vocab_size
assert len(layer_dims) >= 2

C = torch.randn((vocab_size, DimEmb), generator=g)  # 嵌入矩阵：(vocab_size, DimEmb)

n_layers = len(layer_dims) - 1
Ws, bs = [], []
for i in range(n_layers):
    in_dim, out_dim = layer_dims[i], layer_dims[i + 1]
    scale = 0.01 if i == n_layers - 1 else 0.2  # 输出层 0.01，隐藏层 0.2
    Ws.append(torch.randn((in_dim, out_dim), generator=g) * scale)
    bs.append(torch.zeros(out_dim))
#深层参数矩阵采用缩放初始化

parameters = [C] + Ws + bs

max_steps = 20000     # 训练步数
lr_init = 1         # 初始学习率
lr_schedule = 'exponential'  # constant / step / exponential / cosine / linear

In [6]:
# 用前 LenContext 个字预测下一个字
# X: (样本数, LenContext)，Y: (样本数,)
X_flat, Y = [], []
for w in words:
    ctx = [0] * LenContext
    for c in w + '.':
        X_flat.extend(ctx)
        Y.append(stoi[c])
        ctx = ctx[1:] + [stoi[c]]
X = torch.tensor(X_flat, dtype=torch.long).view(-1, LenContext)
Y = torch.tensor(Y, dtype=torch.long)

# 参数默认不需要梯度，这里统一打开
for p in parameters:
    p.requires_grad_(True)

for i in range(max_steps):
  
  # 1. 取一个 batch
  ix = torch.randint(0, X.shape[0], (32,))
  
  # 2. 查嵌入表，再展平
  emb = C[X[ix]] # (32, LenContext, DimEmb)
  h = emb.view(-1, DimEmb*LenContext) # (32, DimEmb*LenContext)
  
  # 3. 隐藏层：线性 + tanh
  for W, b in zip(Ws[:-1], bs[:-1]):
    h = torch.tanh(h @ W + b)
  
  # 4. 输出层：线性，得到 vocab_size 个类别的分数
  logits = h @ Ws[-1] + bs[-1] # (32, vocab_size)
  loss = F.cross_entropy(logits, Y[ix])
  
  # 5. 清空旧梯度，反向传播
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # 6. 学习率调度：constant / step / exponential / cosine / linear
  if lr_schedule == 'constant':
    lr = lr_init
  elif lr_schedule == 'step':
    lr = lr_init * (0.01 ** (i // (max_steps // 5)))  # 每 max_steps/10 步乘 0.01
  elif lr_schedule == 'exponential':
    lr = lr_init * (0.01 ** (i / max_steps))  # 指数衰减到 0.01
  elif lr_schedule == 'cosine':
    lr = 0.5 * lr_init * (1 + math.cos(math.pi * i / max_steps))  # 余弦退火到 0
  elif lr_schedule == 'linear':
    lr = lr_init * (1 - i / max_steps)  # 线性衰减到 0
  else:
    raise ValueError(f'未知的 lr_schedule: {lr_schedule}')
  
  for p in parameters:
    p.data += -lr * p.grad

  print(i, ' : ', loss.item())

0  :  8.806114196777344
1  :  8.783154487609863
2  :  8.46584415435791
3  :  8.616085052490234
4  :  8.76832389831543
5  :  8.424188613891602
6  :  7.215358257293701
7  :  7.612551689147949
8  :  7.075096607208252
9  :  7.595662593841553
10  :  8.688010215759277
11  :  8.191950798034668
12  :  6.859597206115723
13  :  8.012086868286133
14  :  7.797136306762695
15  :  7.4217753410339355
16  :  7.010962963104248
17  :  7.2789459228515625
18  :  7.093048095703125
19  :  7.0614542961120605
20  :  7.432518005371094
21  :  7.502587795257568
22  :  7.043704509735107
23  :  6.810166358947754
24  :  7.28183650970459
25  :  8.796252250671387
26  :  7.034759044647217
27  :  9.10522174835205
28  :  7.965376853942871
29  :  7.149633407592773
30  :  7.252014636993408
31  :  7.401247978210449
32  :  6.544717311859131
33  :  6.339784145355225
34  :  7.142520904541016
35  :  5.969780921936035
36  :  6.169149875640869
37  :  6.442607402801514
38  :  6.678048133850098
39  :  8.005810737609863
40  :  6.26

In [8]:
@torch.no_grad()
def generate(max_len=3):
    out = []
    context = [0] * LenContext  # 初始上下文：LenContext 个边界符

    for _ in range(max_len):

        # 1. 上下文 → 嵌入 → 展平
        emb = C[torch.tensor([context])] # (1, LenContext, DimEmb)
        h = emb.view(1, -1) # (1, DimEmb*LenContext)

        # 2. 隐藏层：线性 + tanh
        for W, b in zip(Ws[:-1], bs[:-1]):
            h = torch.tanh(h @ W + b)

        # 3. 输出层：线性，得到 vocab_size 个类别的分数
        logits = h @ Ws[-1] + bs[-1] # (1, vocab_size)

        # 4. softmax 得到下一个字的概率
        probs = F.softmax(logits, dim=1)[0]

        # 5. 姓名至少 2 个字：前两步禁止采到结束符 0
        if len(out) < 2:
            probs = probs.clone()
            probs[0] = 0
            probs /= probs.sum()

        # 6. 按概率采样下一个字
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()

        # 7. 采到结束符就停止
        if ix == 0:
            break

        # 8. 记录这个字，并把上下文窗口右移一格
        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

for _ in range(20):
    print(generate())

裴欣飞
赵红嘉
耿柏泽
陈遨法
曹思霏
卢玉建
彭甜
辛树寿
孔孙德
茅祥毓
帅鼙
温万垚
干锦冉
李敏璇
钟卫
李欣鹃
俞姬
禹墨
顾博成
柯丁
